# 🔬 A Planning Agent for Deep Research & Structured Report Generation

## Learning Objectives
In this notebook, you will learn:
1. **Planning a report** - Let the model research a topic briefly, then plan the report's sections as structured output
2. **A nested section-builder workflow** - Query generation → web search → section writing, compiled as its own graph
3. **Dynamic fan-out with `Send`** - Dispatch one section builder per planned section, in parallel, and gather the results
4. **Synthesis sections** - Write the introduction and conclusion from the researched sections, then compile the report
5. **Keeping research affordable** - The knobs (queries, results, sections, source length) that bound search and token cost

## Where this fits
`01_planning_react_vs_plan.ipynb` and `02_planning_decompose_execute_replan.ipynb` were single-agent workflows: one plan of lookups, executed one step at a time. Here the plan is a **report outline**, and each planned section is handed to a **nested workflow** (its own compiled graph) - so this is a **compound agent**: one overall system containing nested workflows, not a team of independent agents talking to each other. The fan-out mechanics are module 08's orchestrator-worker pattern (`modules/08_orchestrator_worker/notebooks/01_orchestrator_worker_send_api.ipynb`); the new part is that a *planner* decides what the workers do. `04_reflective_dynamic_planning_agent.ipynb` then nests a whole ReAct agent inside a plan-and-replan loop.

## Prerequisites
- `Send` and fan-out/fan-in reducers - `modules/08_orchestrator_worker/notebooks/01_orchestrator_worker_send_api.ipynb`
- `with_structured_output` - `modules/04_langchain_agents_and_middleware/notebooks/03_structured_output.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` and `TAVILY_API_KEY` in the project-root `.env`

> **Cost note**: every run issues real Tavily searches. The defaults in Step 2 keep one run to **six searches** (about a minute). Raise them for a fuller report, and watch your Tavily quota if you are on the free tier.

---
## 📖 Part 1: The Deep-Research Planning Agent

![](https://i.imgur.com/STSC73k.png)

This project focuses on building a **Planning Agent for Deep Research and Structured Report Generation**. The agent automates the process of analyzing a user-defined topic, performing web research, and generating a well-structured report. The workflow includes the following components:

1. **Report Planning**:
   - The agent analyzes the user-provided **topic** and **default report template** to create a custom plan for the report.
   - Sections such as **Introduction**, **Key Sections**, and **Conclusion** are defined based on the topic.
   - A **web search tool** is used to collect information required before deciding the main sections.

2. **Parallel Execution for Research and Writing**:
   - The agent uses **parallel execution** to efficiently perform:
     - **Web Research**: Queries are generated for each section and executed via the web search tool to retrieve up-to-date information.
     - **Section Writing**: The retrieved data is used to write content for each section, with the following process:
       - The **Researcher** gathers relevant data from the web.
       - The **Section Writer** uses the data to generate structured content for the assigned section.

3. **Formatting Completed Sections**:
   - Once all sections are written, they are formatted to ensure consistency and adherence to the report structure.

4. **Introduction and Conclusion Writing**:
   - After the main sections are completed and formatted:
     - The **Introduction** and **Conclusion** are written based on the content of the remaining sections (in parallel)
     - This process ensures that these sections align with the overall flow and insights of the report.

5. **Final Compilation**:
   - All completed sections are compiled together to generate the **final report**.
   - The final output is a comprehensive and structured document in the style of wiki docs.

> **Key Insight**: the planner does not answer the question - it decides *which pieces of work exist*. Everything after it is sized by that decision, which is why the cost knobs below cap the plan.

---
## 📦 Step 1: Environment Setup

Load the API keys and create the chat model through the course factory.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY and TAVILY_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: Planner, query generator and section writers
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

## 🎚️ Step 2: Cost Knobs

A research agent's cost is multiplicative: *sections × queries per section × results per query × tokens per result*. These constants bound every factor. The original version of this notebook used 3 queries, 6 results per query, 4,000 tokens of raw page per result and no cap on sections - tens of searches and well over 100k tokens per run.

In [ ]:
# ============================================================================
# COST KNOBS: Everything that multiplies searches and tokens
# ============================================================================
PLANNING_QUERIES = 2        # searches used to plan the outline
QUERIES_PER_SECTION = 2     # searches per researched section
RESULTS_PER_QUERY = 3       # Tavily results per search
MAX_RESEARCH_SECTIONS = 2   # researched sections kept from the plan (intro/conclusion are extra)
SOURCE_TOKENS = 800         # raw page tokens kept per source for section writing
SEARCH_DEPTH = "basic"      # "advanced" costs 2 Tavily credits per search

max_searches = PLANNING_QUERIES + MAX_RESEARCH_SECTIONS * QUERIES_PER_SECTION
print(f"✅ At most {max_searches} Tavily searches per report")

---
## 🧱 Part 2: State Schemas and Search Utilities

Each set of operations (nodes) has its own schema. `ReportState` is the outer graph's state; `SectionState` belongs to the nested section builder. `completed_sections` uses an `operator.add` reducer, so the results of parallel section builders are **appended** rather than overwriting each other.

In [ ]:
# ============================================================================
# SCHEMAS: Structured outputs and graph states
# ============================================================================
import operator
from typing import Annotated, List

from pydantic import BaseModel, Field
from typing_extensions import TypedDict


class Section(BaseModel):
    name: str = Field(description="Name for a particular section of the report.")
    description: str = Field(description="Brief overview of the main topics and concepts to be covered in this section.")
    research: bool = Field(description="Whether to perform web search for this section of the report.")
    content: str = Field(description="The content for this section.")


class Sections(BaseModel):
    sections: List[Section] = Field(description="All the Sections of the overall report.")


class SearchQuery(BaseModel):
    search_query: str = Field(description="Query for web search.")


class Queries(BaseModel):
    queries: List[SearchQuery] = Field(description="List of web search queries.")


class ReportStateInput(TypedDict):
    topic: str  # Report topic


class ReportStateOutput(TypedDict):
    final_report: str  # Final report


class ReportState(TypedDict):
    topic: str                                          # Report topic
    sections: list[Section]                             # The planned sections, in report order
    completed_sections: Annotated[list, operator.add]   # Filled in parallel via Send()
    report_sections_from_research: str                  # Researched sections, as context for intro/conclusion
    final_report: str                                   # Final report


class SectionState(TypedDict):
    section: Section                        # Report section
    search_queries: list[SearchQuery]       # List of search queries
    source_str: str                         # Formatted source content from web search
    report_sections_from_research: str      # Context for the final (non-research) sections
    completed_sections: list[Section]       # Duplicated in the outer state for Send()


class SectionOutputState(TypedDict):
    completed_sections: list[Section]


print("✅ Schemas defined")

### 2.1 🔎 Running Searches in Parallel

`run_search_queries(...)` runs a list of Tavily searches **concurrently** with `asyncio.gather` and the async Tavily client. A failed search is dropped instead of failing the whole batch. A module-level counter records how many searches a run really made, so the cost claim in Step 2 can be checked.

In [ ]:
# ============================================================================
# SEARCH UTILITY: Concurrent Tavily searches with a call counter
# ============================================================================
import asyncio
from typing import Dict, Union

from tavily import AsyncTavilyClient

tavily_client = AsyncTavilyClient()  # reads TAVILY_API_KEY
search_log: list[str] = []           # every query actually sent


async def run_search_queries(
    search_queries: List[Union[str, SearchQuery]],
    num_results: int = RESULTS_PER_QUERY,
    include_raw_content: bool = False,
) -> List[Dict]:
    """Run several Tavily searches concurrently; return the successful responses."""
    query_strs = [q.search_query if isinstance(q, SearchQuery) else str(q) for q in search_queries]
    search_log.extend(query_strs)
    tasks = [
        tavily_client.search(
            query=q,
            max_results=num_results,
            search_depth=SEARCH_DEPTH,
            include_answer=False,
            include_raw_content=include_raw_content,
        )
        for q in query_strs
    ]
    responses = await asyncio.gather(*tasks, return_exceptions=True)
    for q, r in zip(query_strs, responses):
        if isinstance(r, Exception):
            print(f"⚠️  search failed for '{q}': {r}")
    return [r for r in responses if not isinstance(r, Exception)]


print("✅ run_search_queries ready")

`format_search_query_results(...)` extracts the context from the Tavily results, deduplicates sources by URL, and formats each as Source / URL / most relevant content - optionally with the raw page text truncated to a token budget, to keep prompts within bounds.

In [ ]:
# ============================================================================
# FORMAT UTILITY: Deduplicate sources and truncate raw content to a token budget
# ============================================================================
import tiktoken
from typing import Any, Dict, List, Union

def format_search_query_results(
    search_response: Union[Dict[str, Any], List[Any]],
    max_tokens: int = 500,
    include_raw_content: bool = False
) -> str:
    encoding = tiktoken.encoding_for_model("gpt-4")
    sources_list = []

    # Handle different response formats
    # if search results is a dict
    if isinstance(search_response, dict):
        if 'results' in search_response:
            sources_list.extend(search_response['results'])
        else:
            sources_list.append(search_response)
    # if search results is a list
    elif isinstance(search_response, list):
        for response in search_response:
            if isinstance(response, dict):
                if 'results' in response:
                    sources_list.extend(response['results'])
                else:
                    sources_list.append(response)
            elif isinstance(response, list):
                sources_list.extend(response)

    if not sources_list:
        return "No search results found."

    # Deduplicate by URL and keep unique sources (website urls)
    unique_sources = {}
    for source in sources_list:
        if isinstance(source, dict) and 'url' in source:
            if source['url'] not in unique_sources:
                unique_sources[source['url']] = source

    # Format output
    formatted_text = "Content from web search:\n\n"
    for i, source in enumerate(unique_sources.values(), 1):
        formatted_text += f"Source {source.get('title', 'Untitled')}:\n===\n"
        formatted_text += f"URL: {source['url']}\n===\n"
        formatted_text += f"Most relevant content from source: {source.get('content', 'No content available')}\n===\n"

        if include_raw_content:
            # truncate raw webpage content to a certain number of tokens to prevent exceeding LLM max token window
            raw_content = source.get("raw_content", "")
            if raw_content:
                tokens = encoding.encode(raw_content)
                truncated_tokens = tokens[:max_tokens]
                truncated_content = encoding.decode(truncated_tokens)
                formatted_text += f"Raw Content: {truncated_content}\n\n"

    return formatted_text.strip()


print("✅ format_search_query_results ready")

In [ ]:
# ============================================================================
# TRY THE UTILITIES: One search, formatted (raw content truncated to 200 tokens)
# ============================================================================
docs = await run_search_queries(["langgraph"], num_results=2, include_raw_content=True)
output = format_search_query_results(docs, max_tokens=200, include_raw_content=True)
print(output[:1500])

---
## 📋 Part 3: Planning the Report

The planner is where this agent *plans*. It works in two structured-output calls:

- **`REPORT_PLAN_QUERY_GENERATOR_PROMPT`**: generate a few search queries about the topic, so the outline is based on current information rather than the model's memory alone.
- **`REPORT_PLAN_SECTION_GENERATOR_PROMPT`**: given the default report template, the topic and those search results, produce the list of sections - each with a name, description, whether it needs research, and empty content.

The default template below is the starting point the LLM adapts into a custom structure.

In [ ]:
# ============================================================================
# DEFAULT REPORT TEMPLATE
# ============================================================================
# Structure
DEFAULT_REPORT_STRUCTURE = """The report structure should focus on breaking-down the user-provided topic
                              and building a comprehensive report in markdown using the following format:

                              1. Introduction (no web search needed)
                                    - Brief overview of the topic area

                              2. Main Body Sections:
                                    - Each section should focus on a sub-topic of the user-provided topic
                                    - Include any key concepts and definitions
                                    - Provide real-world examples or case studies where applicable

                              3. Conclusion (no web search needed)
                                    - Aim for 1 structural element (either a list of table) that distills the main body sections
                                    - Provide a concise summary of the report

                              When generating the final response in markdown, if there are special characters in the text,
                              such as the dollar symbol, ensure they are escaped properly for correct rendering e.g $25.5 should become \$25.5
                          """

print("✅ Template ready")

In [ ]:
# ============================================================================
# PLANNER PROMPTS: Planning queries, then the section outline
# ============================================================================
REPORT_PLAN_QUERY_GENERATOR_PROMPT = """You are an expert technical report writer, helping to plan a report.

The report will be focused on the following topic:
{topic}

The report structure will follow these guidelines:
{report_organization}

Your goal is to generate {number_of_queries} search queries that will help gather comprehensive information for planning the report sections.

The query should:
1. Be related to the topic
2. Help satisfy the requirements specified in the report organization

Make the query specific enough to find high-quality, relevant sources while covering the depth and breadth needed for the report structure.
"""

REPORT_PLAN_SECTION_GENERATOR_PROMPT = """You are an expert technical report writer, helping to plan a report.

Your goal is to generate the outline of the sections of the report.

The overall topic of the report is:
{topic}

The report should follow this organizational structure:
{report_organization}

You should reflect on this additional context information from web searches to plan the main sections of the report:
{search_context}

Now, generate the sections of the report. Each section should have the following fields:
- Name - Name for this section of the report.
- Description - Brief overview of the main topics and concepts to be covered in this section.
- Research - Whether to perform web search for this section of the report or not.
- Content - The content of the section, which you will leave blank for now.

Consider which sections require web search.
For example, introduction and conclusion will not require research because they will distill information from other parts of the report.
"""

print("✅ Planner prompts ready")

![](https://i.imgur.com/54Jyv71.png)

The planner node runs the two calls with a search in between. After the LLM returns the outline, **code** enforces the cost cap: only the first `MAX_RESEARCH_SECTIONS` researched sections are kept (the non-research introduction and conclusion always stay). A prompt can *ask* for fewer sections; only code can guarantee it.

In [ ]:
# ============================================================================
# PLANNER NODE: Research the topic briefly, then plan the sections
# ============================================================================
from langchain_core.messages import HumanMessage, SystemMessage


def cap_research_sections(sections: list[Section], limit: int) -> list[Section]:
    """Keep every non-research section and the first `limit` research sections, in order."""
    kept, n_research = [], 0
    for s in sections:
        if s.research:
            if n_research == limit:
                continue
            n_research += 1
        kept.append(s)
    return kept


async def generate_report_plan(state: ReportState):
    """Generate the overall plan for building the report"""
    topic = state["topic"]
    print("--- 📋 Generating Report Plan ---")

    queries = llm.with_structured_output(Queries).invoke([
        SystemMessage(content=REPORT_PLAN_QUERY_GENERATOR_PROMPT.format(
            topic=topic, report_organization=DEFAULT_REPORT_STRUCTURE, number_of_queries=PLANNING_QUERIES)),
        HumanMessage(content="Generate search queries that will help with planning the sections of the report."),
    ])
    query_list = [q.search_query for q in queries.queries][:PLANNING_QUERIES]

    search_docs = await run_search_queries(query_list, include_raw_content=False)
    search_context = format_search_query_results(search_docs, include_raw_content=False) if search_docs else "No search results available."

    report_sections = llm.with_structured_output(Sections).invoke([
        SystemMessage(content=REPORT_PLAN_SECTION_GENERATOR_PROMPT.format(
            topic=topic, report_organization=DEFAULT_REPORT_STRUCTURE, search_context=search_context)),
        HumanMessage(content="Generate the sections of the report. Your response must include a 'sections' field "
                             "containing a list of sections. Each section must have: name, description, research, and content fields."),
    ])

    sections = cap_research_sections(report_sections.sections, MAX_RESEARCH_SECTIONS)
    print(f"--- 📋 Planned {len(report_sections.sections)} sections, kept {len(sections)}: ---")
    for s in sections:
        print(f"      {'🔎' if s.research else '✍️ '} {s.name}")
    return {"sections": sections}


print("✅ generate_report_plan defined")

---
## 🧩 Part 4: The Nested Section-Builder Workflow

Each researched section is built by the same three-step workflow:

1. **`generate_queries`** - `REPORT_SECTION_QUERY_GENERATOR_PROMPT` turns the section description into targeted search queries.
2. **`search_web`** - runs those queries and formats the results (with truncated raw page content).
3. **`write_section`** - `SECTION_WRITER_PROMPT` constrains the writer on style, structure, length and sources.

The three steps are compiled into their **own graph**, which the outer graph then uses as a single node - that nesting is what makes this a compound agent.

In [ ]:
# ============================================================================
# SECTION-BUILDER PROMPTS: Query generation and section writing
# ============================================================================
REPORT_SECTION_QUERY_GENERATOR_PROMPT = """Your goal is to generate targeted web search queries that will gather comprehensive information for writing a technical report section.

Topic for this section:
{section_topic}

When generating {number_of_queries} search queries, ensure that they:
1. Cover different aspects of the topic (e.g., core features, real-world applications, technical architecture)
2. Include specific technical terms related to the topic
3. Target recent information by including year markers where relevant (e.g., "2024")
4. Look for comparisons or differentiators from similar technologies/approaches
5. Search for both official documentation and practical implementation examples

Your queries should be:
- Specific enough to avoid generic results
- Technical enough to capture detailed implementation information
- Diverse enough to cover all aspects of the section plan
- Focused on authoritative sources (documentation, technical blogs, academic papers)"""

SECTION_WRITER_PROMPT = """You are an expert technical writer crafting one specific section of a technical report.

Title for the section:
{section_title}

Topic for this section:
{section_topic}

Guidelines for writing:

1. Technical Accuracy:
- Include specific version numbers
- Reference concrete metrics/benchmarks
- Cite official documentation
- Use technical terminology precisely

2. Length and Style:
- Strict 150-200 word limit
- No marketing language
- Technical focus
- Write in simple, clear language do not use complex words unnecessarily
- Start with your most important insight in **bold**
- Use short paragraphs (2-3 sentences max)

3. Structure:
- Use ## for section title (Markdown format)
- Only use ONE structural element IF it helps clarify your point:
  * Either a focused table comparing 2-3 key items (using Markdown table syntax)
  * Or a short list (3-5 items) using proper Markdown list syntax:
    - Use `*` or `-` for unordered lists
    - Use `1.` for ordered lists
    - Ensure proper indentation and spacing
- End with ### Sources that references the below source material formatted as:
  * List each source with title, date, and URL
  * Format: `- Title : URL`

3. Writing Approach:
- Include at least one specific example or case study if available
- Use concrete details over general statements
- Make every word count
- No preamble prior to creating the section content
- Focus on your single most important point

4. Use this source material obtained from web searches to help write the section:
{context}

5. Quality Checks:
- Format should be Markdown
- Exactly 150-200 words (excluding title and sources)
- Careful use of only ONE structural element (table or bullet list) and only if it helps clarify your point
- One specific example / case study if available
- Starts with bold insight
- No preamble prior to creating the section content
- Sources cited at end
- If there are special characters in the text, such as the dollar symbol,
  ensure they are escaped properly for correct rendering e.g $25.5 should become \$25.5
"""

print("✅ Section prompts ready")

In [ ]:
# ============================================================================
# SECTION-BUILDER NODES: generate_queries -> search_web -> write_section
# ============================================================================
def generate_queries(state: SectionState):
    """Generate search queries for a specific report section"""
    section = state["section"]
    print(f"--- 🔎 Generating search queries for: {section.name} ---")
    search_queries = llm.with_structured_output(Queries).invoke([
        SystemMessage(content=REPORT_SECTION_QUERY_GENERATOR_PROMPT.format(
            section_topic=section.description, number_of_queries=QUERIES_PER_SECTION)),
        HumanMessage(content="Generate search queries on the provided topic."),
    ])
    return {"search_queries": search_queries.queries[:QUERIES_PER_SECTION]}


async def search_web(state: SectionState):
    """Search the web for each query and return a formatted string of sources."""
    search_docs = await run_search_queries(state["search_queries"], include_raw_content=True)
    return {"source_str": format_search_query_results(search_docs, max_tokens=SOURCE_TOKENS, include_raw_content=True)}


def write_section(state: SectionState):
    """Write a section of the report"""
    section = state["section"]
    print(f"--- ✍️  Writing section: {section.name} ---")
    section_content = llm.invoke([
        SystemMessage(content=SECTION_WRITER_PROMPT.format(
            section_title=section.name, section_topic=section.description, context=state["source_str"])),
        HumanMessage(content="Generate a report section based on the provided sources."),
    ])
    # Return an updated copy - parallel branches should not mutate shared state objects
    return {"completed_sections": [section.model_copy(update={"content": section_content.content})]}


print("✅ Section-builder nodes defined")

![](https://i.imgur.com/5VEYGrQ.png)

In [ ]:
# ============================================================================
# NESTED GRAPH: Compile the section builder as its own graph
# ============================================================================
from langgraph.graph import END, START, StateGraph

section_builder = StateGraph(SectionState, output_schema=SectionOutputState)
section_builder.add_node("generate_queries", generate_queries)
section_builder.add_node("search_web", search_web)
section_builder.add_node("write_section", write_section)

section_builder.add_edge(START, "generate_queries")
section_builder.add_edge("generate_queries", "search_web")
section_builder.add_edge("search_web", "write_section")
section_builder.add_edge("write_section", END)
section_builder_subagent = section_builder.compile()

print("🧩 Nested graph nodes:", [n for n in section_builder_subagent.get_graph().nodes if not n.startswith("__")])

### 4.1 🔀 Dynamic Fan-Out with `Send`

The planner decided how many researched sections exist, so the graph cannot have a fixed edge per section. `parallelize_section_writing` returns one `Send("section_builder_with_web_search", {"section": s})` per researched section; LangGraph runs those nested workflows in parallel and the `operator.add` reducer gathers their `completed_sections`.

In [ ]:
# ============================================================================
# FAN-OUT: One nested section builder per researched section
# ============================================================================
from langgraph.types import Send


def parallelize_section_writing(state: ReportState):
    """The "map" step: start web research + writing for every researched section in parallel"""
    return [Send("section_builder_with_web_search", {"section": s}) for s in state["sections"] if s.research]


print("✅ parallelize_section_writing defined")

---
## ✍️ Part 5: Synthesis Sections and the Final Report

Once the researched sections are done, they are formatted into one context string. The introduction and conclusion - the sections the planner marked `research=False` - are then written **from that context** (in parallel, again with `Send`), using `FINAL_SECTION_WRITER_PROMPT`. Finally `compile_final_report` puts every section back in the planned order.

![](https://i.imgur.com/6e5ZWK4.png)

In [ ]:
# ============================================================================
# FORMAT COMPLETED SECTIONS: Context for the introduction and conclusion
# ============================================================================
def format_sections(sections: list[Section]) -> str:
    """ Format a list of report sections into a single text string """
    formatted_str = ""
    for idx, section in enumerate(sections, 1):
        formatted_str += f"""
{'='*60}
Section {idx}: {section.name}
{'='*60}
Description:
{section.description}
Requires Research:
{section.research}

Content:
{section.content if section.content else '[Not yet written]'}

"""
    return formatted_str


def format_completed_sections(state: ReportState):
    """ Gather completed sections from research and format them as context for writing the final sections """
    print("--- 🧾 Formatting completed sections ---")
    return {"report_sections_from_research": format_sections(state["completed_sections"])}


print("✅ format_completed_sections defined")

In [ ]:
# ============================================================================
# FINAL-SECTION PROMPT: Introduction and conclusion
# ============================================================================
FINAL_SECTION_WRITER_PROMPT = """You are an expert technical writer crafting a section that synthesizes information from the rest of the report.

Title for the section:
{section_title}

Topic for this section:
{section_topic}

Available report content of already completed sections:
{context}

1. Section-Specific Approach:

For Introduction:
- Use # for report title (Markdown format)
- 50-100 word limit
- Write in simple and clear language
- Focus on the core motivation for the report in 1-2 paragraphs
- Use a clear narrative arc to introduce the report
- Include NO structural elements (no lists or tables)
- No sources section needed

For Conclusion/Summary:
- Use ## for section title (Markdown format)
- 100-150 word limit
- For comparative reports:
    * Must include a focused comparison table using Markdown table syntax
    * Table should distill insights from the report
    * Keep table entries clear and concise
- For non-comparative reports:
    * Only use ONE structural element IF it helps distill the points made in the report:
    * Either a focused table comparing items present in the report (using Markdown table syntax)
    * Or a short list using proper Markdown list syntax:
      - Use `*` or `-` for unordered lists
      - Use `1.` for ordered lists
      - Ensure proper indentation and spacing
- End with specific next steps or implications
- No sources section needed

3. Writing Approach:
- Use concrete details over general statements
- Make every word count
- Focus on your single most important point

4. Quality Checks:
- For introduction: 50-100 word limit, # for report title, no structural elements, no sources section
- For conclusion: 100-150 word limit, ## for section title, only ONE structural element at most, no sources section
- Markdown format
- Do not include word count or any preamble in your response
- If there are special characters in the text, such as the dollar symbol,
  ensure they are escaped properly for correct rendering e.g $25.5 should become \$25.5"""

print("✅ Final-section prompt ready")

![](https://i.imgur.com/pRv4PX8.png)

In [ ]:
# ============================================================================
# FINAL SECTIONS: Write intro/conclusion from the researched sections, in parallel
# ============================================================================
def write_final_sections(state: SectionState):
    """ Write the final sections of the report, which do not require web search and use the completed sections as context"""
    section = state["section"]
    print(f"--- ✍️  Writing final section: {section.name} ---")
    section_content = llm.invoke([
        SystemMessage(content=FINAL_SECTION_WRITER_PROMPT.format(
            section_title=section.name, section_topic=section.description,
            context=state["report_sections_from_research"])),
        HumanMessage(content="Craft a report section based on the provided sources."),
    ])
    return {"completed_sections": [section.model_copy(update={"content": section_content.content})]}


def parallelize_final_section_writing(state: ReportState):
    """ Write any final sections using the Send API to parallelize the process """
    return [
        Send("write_final_sections", {"section": s, "report_sections_from_research": state["report_sections_from_research"]})
        for s in state["sections"]
        if not s.research
    ]


print("✅ Final-section nodes defined")

![](https://i.imgur.com/wLxCNZ5.png)

In [ ]:
# ============================================================================
# COMPILE THE REPORT: Put the written sections back in planned order
# ============================================================================
def compile_final_report(state: ReportState):
    """ Compile the final report """
    print("--- 📚 Compiling final report ---")
    content_by_name = {s.name: s.content for s in state["completed_sections"]}
    all_sections = "\n\n".join(content_by_name.get(s.name, f"## {s.name}\n[missing]") for s in state["sections"])

    # Escape unescaped $ symbols to display properly in Markdown
    formatted = all_sections.replace("\\$", "TEMP_PLACEHOLDER")  # temporarily mark already escaped $
    formatted = formatted.replace("$", "\\$")                     # escape all $
    formatted = formatted.replace("TEMP_PLACEHOLDER", "\\$")      # restore originally escaped $
    return {"final_report": formatted}


print("✅ compile_final_report defined")

---
## 🏗️ Part 6: Building and Running the Planning Agent

Bring every component and the nested section builder together. Note the two `Send` fan-outs: `generate_report_plan → section_builder_with_web_search` and `format_completed_sections → write_final_sections`.

![](https://i.imgur.com/STSC73k.png)

In [ ]:
# ============================================================================
# OUTER GRAPH: plan -> fan out section builders -> format -> fan out intro/conclusion -> compile
# ============================================================================
builder = StateGraph(ReportState, input_schema=ReportStateInput, output_schema=ReportStateOutput)

builder.add_node("generate_report_plan", generate_report_plan)
builder.add_node("section_builder_with_web_search", section_builder_subagent)  # a compiled graph as a node
builder.add_node("format_completed_sections", format_completed_sections)
builder.add_node("write_final_sections", write_final_sections)
builder.add_node("compile_final_report", compile_final_report)

builder.add_edge(START, "generate_report_plan")
builder.add_conditional_edges("generate_report_plan", parallelize_section_writing, ["section_builder_with_web_search"])
builder.add_edge("section_builder_with_web_search", "format_completed_sections")
builder.add_conditional_edges("format_completed_sections", parallelize_final_section_writing, ["write_final_sections"])
builder.add_edge("write_final_sections", "compile_final_report")
builder.add_edge("compile_final_report", END)

reporter_agent = builder.compile()

g = reporter_agent.get_graph(xray=True)
print("🧩 Nodes (xray shows the nested graph):", [n for n in g.nodes if not n.startswith("__")])

### 6.1 🚀 Run the Agent

One topic, with the cost knobs from Step 2. The printed lines show the plan, then the parallel section builders interleaving, then the synthesis sections. The run ends by checking the search count against the bound.

In [ ]:
# ============================================================================
# RUN: Plan, research, write and compile one report
# ============================================================================
import time

from IPython.display import Markdown, display

search_log.clear()
topic = "How to build agentic AI systems: design patterns and current frameworks"

start = time.perf_counter()
result = await reporter_agent.ainvoke({"topic": topic}, config={"recursion_limit": 50})
elapsed = time.perf_counter() - start

print(f"\n📊 {len(search_log)} Tavily searches (bound: {max_searches}), {elapsed:.0f}s")
assert len(search_log) <= max_searches
display(Markdown(result["final_report"]))

To research other topics (for example *"How is NVIDIA winning the game against its competitors"* or *"How DeepSeek has disrupted the AI market"*), change `topic` and re-run the cell. For a longer report, raise `MAX_RESEARCH_SECTIONS` and `QUERIES_PER_SECTION` in Step 2 - the printed bound tells you how many searches the run may use.

---
## 📝 Summary

In this notebook, we learned:

### 1. Planning a report
- The planner researches briefly, then emits the outline as structured `Sections` - it decides *what work exists*, not the answer.

### 2. A nested section-builder workflow
- Query generation → search → writing is compiled as its own graph and used as one node of the outer graph - a compound agent, not a multi-agent team.

### 3. Dynamic fan-out with `Send`
- One `Send` per planned section runs the nested workflows in parallel; an `operator.add` reducer gathers their results. Nodes return updated copies instead of mutating shared objects.

### 4. Synthesis sections
- The introduction and conclusion are written from the researched sections, then `compile_final_report` restores the planned order.

### 5. Keeping research affordable
- Cost multiplies across sections, queries, results and source length. Cap the plan **in code** and count real searches to check the bound.

### Next Steps
- **`04_reflective_dynamic_planning_agent.ipynb`**: a planner and replanner that delegate each step to a nested ReAct research agent and revise the plan after every step